In [ ]:
"""
Memory experiment analysis: min-distance + ROC curves.
Refactored into a single organized script. 
Modify only `run_experiment_at_noise` to explore new dynamics.
"""
# ===================== Imports =====================
import argparse, sys, os, glob, json, math, datetime, torch, yaml
import matplotlib.pyplot as plt, numpy as np, pandas as pd

from collections import defaultdict
from scipy.stats import norm
from sklearn.metrics import roc_curve, auc
from sklearn.linear_model import LinearRegression
from sklearn.utils import resample


# project-specific paths
sys.path.append('/orcd/data/jhm/001/om2/jmhicks/projects/TextureStreaming/code/')
sys.path.append('../utls/')
sys.path.append('../src/model/')
sys.path.append("/orcd/data/jhm/001/om2/bjmedina/auditory-memory/memory/")

from chexture_choolbox.auditorytexture.texture_model import TextureModel
from chexture_choolbox.auditorytexture.helpers import FlattenStats
from texture_prior.params import model_params, statistics_set, texture_dataset
from texture_prior.utils import path

from encoders import *
from utls.plotting import ensure_dir
from utls.loading import load_results_with_exclusion_2, move_sequences_to_used, load_results_with_exclusion_no_dropping
from utls.runners import run_experiment_scores
from utls.analysis_helpers import rocs_across_noise, convert_human_to_model_struct, compute_scaling_vs_human, convert_human_to_model_struct
from utls.analysis_helpers import auroc_to_dprime, compute_model_dprime_curve
from utls.analysis_helpers import roc_for_isi, auroc_to_dprime
from utls.plotting import plot_across_noise, plot_noise_overlays
from utls.io_utils import make_model_save_dir, save_all_figures, save_single_figure, save_runs_summary
from utls.roc_utils import roc_from_arrays 
from utls.runners_utils import *



def make_encoder(cfg):
    """
    Create an encoder model based on cfg.
    cfg fields:
        - encoder_type: 'kell2018', 'resnet50', 'texture_pca'
        - model_name, layer, task (for neural nets)
        - statistics_dict, pc_dims, model_params (for PCA texture model)
        - sr, rms_level, duration, device
    """
    etype = cfg['encoder_type']
    
    if etype == 'kell2018':
        return Kell2018Encoder(
            model_name=cfg['model_name'],
            layer=cfg['layer'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )
    
    elif etype == 'resnet50':
        return ResNet50Encoder(
            model_name=cfg['model_name'],
            layer=cfg['layer'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )
    
    elif etype == 'texture_pca':
        return AudioTextureEncoderPCA(
            statistics_dict=cfg['statistics_dict'],
            pc_dims=cfg['pc_dims'],
            model_params=cfg['model_params'],
            sr=cfg.get('sr', 20000),
            rms_level=cfg.get('rms_level', 0.05),
            duration=cfg.get('duration', 2.0),
            device=cfg.get('device', 'cuda')
        )

    else:
        raise ValueError(f"Unknown encoder_type: {etype}")

def encode_batch(encoder, sound_list):
    """
    Encode a list of raw audio arrays and return a (N, D) tensor.
    Assumes encoder(sound) returns a tensor or something with .reshape().
    """
    feats = []
    for s in sound_list:
        out = encoder(s)
        if isinstance(out, dict):
            out = out["embedding"]           # modify if your dict key differs
        feats.append(out.reshape(-1))         # flatten everything

    return torch.stack(feats, dim=0)

In [ ]:
## model parameters needed
## PREPARE THE MEMORY MODEL HYPERPARAMETERS

# need to specify which noise levels to search other
NOISE_LEVELS = np.geomspace(0.01, 5, 10)
metric = "cosine"
noise_level_a = 0.0 # std of noise in the first regime of the noise schedule
noise_level_b = 0.0 # std of noise in the second regime of the noise schedule
rate          = 0.0

## experiment selection
is_multi = True # dictates which class of experiments we run the model on.
which_task = 0
which_isi = 16 ## also need to choose which ISI condition we are looking at in the SINGLE isi case

## encoder (representation) selection
cfg = {
    "encoder_type": "kell2018",
    "model_name": "kell2018",
    "layer": "relu3",
    "task": "word_speaker_audioset",
    "sr": 20000,
    "duration": 2.0,
    "rms_level": 0.05,
    "device": "cuda"
}

## also need to specify directories in which you need to save model performance to 

sys.path.append(f'/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')
print(f'/orcd/data/jhm/001/om2/bjmedina/models/cochdnn/model_directories/{cfg['model_name']}_{cfg['task']}/')

In [ ]:
## need to give someone the ability to choose which experiment (stimulus sets) to
## to run the model on

## PREPARE THE HUMAN RESULTS
if not is_multi:
    tasks = ["ind-nature-len120" ,"global-music-len120", "atexts-len120", "nhs-region-len120"]
    base_path = f"/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{{}}/sequences/isi_{which_isi}/len120/"
else:
    tasks = ["env-sounds" ,"glob-music", "atexts", "nhs-region-len120"]
    base_path = "/mindhive/mcdermott/www/mturk_stimuli/bjmedina/{}/sequences/len120_multi/"

which_task = tasks[which_task] # they should be able to pick the index in the above list

seqs_paths = {tasks[0]: "mem_exp_ind-nature_2025", 
              tasks[1]: "global-music-2025-n_80",
              tasks[2]: "mem_exp_atexts_2025",
              tasks[3]: "nhs-region-n_80"}

hr_task_name = {tasks[0]: "Industrial and Nature", 
                tasks[1]: "Globalized Music",
                tasks[2]: "Auditory Textures",
                tasks[3]: " 'Natural History of Song' "}

if not is_multi: 
    ## SINGLE ISI experiments 
    exps, seqs, fnames, skipped_exps, skipped_seqs, skipped_fnames = load_results_with_exclusion_no_dropping(f"/mindhive/mcdermott/www/bjmedina/experiments/bolivia_2025/results/isi_{which_isi}/{which_task}",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=True)

    print(f" --- Running {hr_task_name[which_task]} SINGLE ISI={which_isi} Experiments --- ")

else:
    ## MULTI ISI experiments
    exps, seqs, fnames, skipped_exps, skipped_seqs, skipped_fnames = load_results_with_exclusion_no_dropping(f"/mindhive/mcdermott/www/bjmedina/experiments/{which_task}/results/{which_task}/len120_multi",
                                                        min_dprime=2,
                                                        min_trials=120,
                                                        skip_len60=True,
                                                        verbose=False,
                                                        return_skipped=True)

    print(f" --- Running {hr_task_name[which_task]} MULTI ISI Experiments --- ")

experiment_list = []
stim_base_path = "/".join(base_path.format(seqs_paths[which_task]).split("/")[:-3])
for seq in seqs:
    with open(base_path.format(seqs_paths[which_task]) + seq, 'r') as f:
        data = json.load(f)
    stim_paths = [stim_base_path + "/" + s 
                  for s in data['filenames_order']]

    experiment_list.append(stim_paths)

all_files_ = sorted({fn for seq in experiment_list for fn in seq})
name_to_idx = {fn: i for i, fn in enumerate(all_files_)}

human_runs = []
for passed_experiments in exps:
    human_runs.append(convert_human_to_model_struct(passed_experiments))

if not is_multi: 
    isis_human = [0, which_isi]
else:
    isis_human = [0, 1, 2, 3, 4, 8, 16, 32, 64]



# must be a better way to 
dprimes_human = []
for k in isis_human:
    aucs = []
    for run in human_runs:
        res = roc_for_isi(run, k)
        if res is not None:
            _, _, auc = res
            aucs.append(auroc_to_dprime(auc))
    dprimes_human.append(np.nanmean(aucs))

human_curve = np.array(dprimes_human, dtype=float)
human_curve = human_curve[~np.isnan(human_curve)]

In [ ]:
## need to also choose WHICH representation we are going to 
## test the models on.

## ALSO, it would be interesteing to be able ot use a 
## concatenated representation

In [ ]:
encoder = make_encoder(cfg)
X = encode_batch(encoder, all_files_)

In [ ]:
plt.scatter(isis_human[1:], human_curve)

In [ ]:
# Determine model type from metric
model_name = "DistanceMemoryModel" if metric in {"euclidean", "cosine", "mahalanobis", "manhattan"} else "LikelihoodMemoryModel"